In [1]:
!pip install torch transformers scikit-learn tqdm

In [2]:
import os
import json
import math
import torch
import logging
import random
import numpy as np
from tqdm import tqdm
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import CrossEntropyLoss
from torch.amp import autocast, GradScaler
from torch.utils.data import DataLoader, Dataset, RandomSampler, SequentialSampler, Subset
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup, RobertaConfig, RobertaModel, AutoTokenizer
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, roc_auc_score, average_precision_score, classification_report, confusion_matrix
from collections import defaultdict, Counter

class Args:
    output_dir = "saved_models_unixcoder"
    model_name_or_path = "microsoft/unixcoder-base"
    train_file = "/kaggle/input/datasets/hasanmahmudabdullah/dfgdataset2/dataset_graphcodebert.jsonl"
    code_length = 384
    train_batch_size = 16
    eval_batch_size = 32
    learning_rate = 2e-5
    max_grad_norm = 1.0
    num_train_epochs = 5
    patience = 2
    metric_for_best_model = "accuracy"
    best_model_path = os.path.join(output_dir, "best_model_text_only.bin")
    split_indices_path = os.path.join(output_dir, "unixcoder_text_only_split_indices.json")
    split_summary_path = os.path.join(output_dir, "unixcoder_text_only_split_summary.json")
    history_path = os.path.join(output_dir, "unixcoder_text_only_training_history.json")
    results_path = os.path.join(output_dir, "unixcoder_text_only_results.txt")
    seed = 42
    test_ratio = 0.10
    val_ratio = 0.08
    num_workers = 2
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

args = Args()
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(args.seed)


In [3]:
class SimpleModel(nn.Module):   
    def __init__(self, encoder, config):
        super(SimpleModel, self).__init__()
        self.encoder = encoder
        self.config = config
        self.dropout = nn.Dropout(config.hidden_dropout_prob)
        self.classifier = nn.Linear(config.hidden_size, 2)

    def forward(self, input_ids=None, attention_mask=None, labels=None): 
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = outputs[0] # [Batch, Seq, Hidden]
        
        # Use CLS token for classification
        logits = self.classifier(self.dropout(sequence_output[:, 0, :]))
        prob = F.softmax(logits, dim=-1)

        if labels is not None:
            loss_fct = CrossEntropyLoss()
            loss = loss_fct(logits, labels)
            return loss, prob
        return prob

In [4]:
class SimpleDataset(Dataset):
    def __init__(self, tokenizer, args, file_path):
        self.args = args
        self.tokenizer = tokenizer
        
        with open(file_path, 'r') as f:
            self.lines = f.readlines()
            
    def __len__(self):
        return len(self.lines)

    def __getitem__(self, item):
        line = self.lines[item]
        entry = json.loads(line)
        
        code = entry.get('code', '')
        label = int(entry.get('label', 0)) if entry.get('label') is not None else 0

        tokens_obj = self.tokenizer(
            code, 
            max_length=self.args.code_length, 
            truncation=True, 
            padding='max_length'
        )
        
        return {
            'input_ids': torch.tensor(tokens_obj['input_ids'], dtype=torch.long),
            'attention_mask': torch.tensor(tokens_obj['attention_mask'], dtype=torch.long),
            'label': torch.tensor(label, dtype=torch.long)
        }

In [5]:
tokenizer = AutoTokenizer.from_pretrained(args.model_name_or_path)
full_dataset = SimpleDataset(tokenizer, args, args.train_file)

def load_entries(filepath):
    with open(filepath, 'r', encoding='utf-8') as f:
        return [json.loads(line) for line in f]

entries = load_entries(args.train_file)
assert len(entries) == len(full_dataset), "Dataset size mismatch between parsed entries and dataset"

def infer_source(entry):
    for key in ("source", "dataset", "origin", "project"):
        value = entry.get(key)
        if value is not None and str(value).strip() != "":
            return str(value).strip()
    return "unknown"

def allocate_counts(total_needed, groups, fraction):
    raw = {g: len(v) * fraction for g, v in groups.items()}
    base = {g: int(math.floor(v)) for g, v in raw.items()}
    remainder = total_needed - sum(base.values())
    order = sorted(groups.keys(), key=lambda g: (raw[g] - base[g], len(groups[g])), reverse=True)
    for g in order[:remainder]:
        base[g] += 1
    return base

def stratified_three_way_split(entries, test_ratio=0.10, val_ratio=0.08, seed=42):
    rng = random.Random(seed)
    source_to_indices = defaultdict(list)
    for idx, entry in enumerate(entries):
        source_to_indices[infer_source(entry)].append(idx)

    for indices in source_to_indices.values():
        rng.shuffle(indices)

    total = len(entries)
    target_test = int(round(total * test_ratio))
    target_val = int(round(total * val_ratio))
    target_train = total - target_test - target_val

    test_alloc = allocate_counts(target_test, source_to_indices, test_ratio)
    trainval_groups = {}
    test_indices = []
    for source, indices in source_to_indices.items():
        take = min(test_alloc[source], len(indices))
        test_indices.extend(indices[:take])
        trainval_groups[source] = indices[take:]

    adjusted_val_ratio = val_ratio / (1.0 - test_ratio)
    val_alloc = allocate_counts(target_val, trainval_groups, adjusted_val_ratio)

    val_indices, train_indices = [], []
    for source, indices in trainval_groups.items():
        take = min(val_alloc[source], len(indices))
        val_indices.extend(indices[:take])
        train_indices.extend(indices[take:])

    train_indices = sorted(train_indices)
    val_indices = sorted(val_indices)
    test_indices = sorted(test_indices)

    assert len(train_indices) == target_train, f"Expected train={target_train}, got {len(train_indices)}"
    assert len(val_indices) == target_val, f"Expected val={target_val}, got {len(val_indices)}"
    assert len(test_indices) == target_test, f"Expected test={target_test}, got {len(test_indices)}"

    return train_indices, val_indices, test_indices

train_indices, val_indices, test_indices = stratified_three_way_split(
    entries,
    test_ratio=args.test_ratio,
    val_ratio=args.val_ratio,
    seed=args.seed,
)

train_dataset = Subset(full_dataset, train_indices)
val_dataset = Subset(full_dataset, val_indices)
test_dataset = Subset(full_dataset, test_indices)

os.makedirs(args.output_dir, exist_ok=True)
with open(args.split_indices_path, 'w', encoding='utf-8') as f:
    json.dump({
        'seed': args.seed,
        'train_indices': train_indices,
        'val_indices': val_indices,
        'test_indices': test_indices,
    }, f)

def source_counts(indices):
    counts = Counter(infer_source(entries[i]) for i in indices)
    return dict(sorted(counts.items()))

split_summary = {
    'total': len(entries),
    'train': len(train_indices),
    'val': len(val_indices),
    'test': len(test_indices),
    'train_source_counts': source_counts(train_indices),
    'val_source_counts': source_counts(val_indices),
    'test_source_counts': source_counts(test_indices),
}
with open(args.split_summary_path, 'w', encoding='utf-8') as f:
    json.dump(split_summary, f, indent=2)

print('Dataset Split Results')
print(f"Total: {split_summary['total']}")
print(f"Train: {split_summary['train']}")
print(f"Val: {split_summary['val']}")
print(f"Test: {split_summary['test']}")
print('Train source counts:', split_summary['train_source_counts'])
print('Val source counts:', split_summary['val_source_counts'])
print('Test source counts:', split_summary['test_source_counts'])
def evaluate(model, dataset, args, tag="Eval", quiet=False):
    dataloader = DataLoader(
        dataset,
        sampler=SequentialSampler(dataset),
        batch_size=args.eval_batch_size,
        num_workers=args.num_workers,
        pin_memory=True,
    )
    model.eval()
    all_probs, all_labels = [], []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc=f"Evaluating {tag}", disable=quiet):
            probs = model(
                input_ids=batch['input_ids'].to(args.device),
                attention_mask=batch['attention_mask'].to(args.device)
            )
            all_probs.append(probs.cpu().numpy())
            all_labels.extend(batch['label'].cpu().numpy())

    all_probs = np.concatenate(all_probs, axis=0)
    all_labels = np.array(all_labels)
    all_preds = np.argmax(all_probs, axis=-1)

    acc = accuracy_score(all_labels, all_preds)
    roc_auc = roc_auc_score(all_labels, all_probs[:, 1])
    pr_auc = average_precision_score(all_labels, all_probs[:, 1])
    tn, fp, fn, tp = confusion_matrix(all_labels, all_preds).ravel()

    if not quiet:
        print("" + "=" * 40)
        print(f"RESULTS ({tag})")
        print("=" * 40)
        print(f"Accuracy : {acc:.4%}")
        print(f"ROC-AUC : {roc_auc:.4f}")
        print(f"PR-AUC : {pr_auc:.4f}")
        print(f"FN Count : {fn}")
        print(f"FP Count : {fp}")
        print("-" * 40)
        print(classification_report(all_labels, all_preds, target_names=['Safe', 'Vuln'], digits=4))
        print("Confusion Matrix:")
        print(confusion_matrix(all_labels, all_preds))

    return {
        'probs': all_probs,
        'labels': all_labels,
        'acc': acc,
        'roc_auc': roc_auc,
        'pr_auc': pr_auc,
        'fn': int(fn),
        'fp': int(fp),
    }


def train(model, train_dataset, val_dataset, args):
    train_dataloader = DataLoader(
        train_dataset,
        sampler=RandomSampler(train_dataset),
        batch_size=args.train_batch_size,
        num_workers=args.num_workers,
        pin_memory=True,
        drop_last=True,
    )

    optimizer = AdamW(model.parameters(), lr=args.learning_rate, eps=1e-8)
    total_steps = len(train_dataloader) * args.num_train_epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(total_steps * 0.1),
        num_training_steps=total_steps,
    )
    scaler = GradScaler('cuda', enabled=torch.cuda.is_available())

    best_val_acc = -1.0
    best_epoch = 0
    patience_counter = 0
    history = []

    if not os.path.exists(args.output_dir):
        os.makedirs(args.output_dir)

    for epoch in range(args.num_train_epochs):
        model.train()
        tr_loss = 0.0
        bar = tqdm(train_dataloader, desc=f"Epoch {epoch + 1}")

        for step, batch in enumerate(bar):
            optimizer.zero_grad(set_to_none=True)
            with autocast(device_type='cuda', enabled=torch.cuda.is_available()):
                loss, _ = model(
                    input_ids=batch['input_ids'].to(args.device),
                    attention_mask=batch['attention_mask'].to(args.device),
                    labels=batch['label'].to(args.device)
                )

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), args.max_grad_norm)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            tr_loss += loss.item()
            bar.set_postfix(loss=tr_loss / (step + 1))

        avg_train_loss = tr_loss / len(train_dataloader)
        val_metrics = evaluate(model, val_dataset, args, tag=f"Validation Epoch {epoch + 1}", quiet=False)
        val_acc = val_metrics['acc']
        history.append({
            'epoch': epoch + 1,
            'train_loss': avg_train_loss,
            'val_acc': val_acc,
            'val_roc_auc': val_metrics['roc_auc'],
            'val_pr_auc': val_metrics['pr_auc'],
            'val_fn': val_metrics['fn'],
            'val_fp': val_metrics['fp'],
        })

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = epoch + 1
            patience_counter = 0
            torch.save(model.state_dict(), args.best_model_path)
            print(f"New best model saved at epoch {best_epoch} with validation accuracy {best_val_acc:.4f}")
        else:
            patience_counter += 1
            print(f"No validation improvement. Patience {patience_counter}/{args.patience}")
            if patience_counter >= args.patience:
                print("Early stopping triggered.")
                break

    with open(args.history_path, 'w', encoding='utf-8') as f:
        json.dump(history, f, indent=2)

    return best_epoch, best_val_acc, history


INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/unixcoder-base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/unixcoder-base/5604afdc964f6c53782a6813140ade5216b99006/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/resolve-cache/models/microsoft/unixcoder-base/5604afdc964f6c53782a6813140ade5216b99006/config.json "HTTP/1.1 200 OK"


config.json:   0%|          | 0.00/691 [00:00<?, ?B/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/unixcoder-base/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/unixcoder-base/5604afdc964f6c53782a6813140ade5216b99006/tokenizer_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/resolve-cache/models/microsoft/unixcoder-base/5604afdc964f6c53782a6813140ade5216b99006/tokenizer_config.json "HTTP/1.1 200 OK"


tokenizer_config.json: 0.00B [00:00, ?B/s]

INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/microsoft/unixcoder-base/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/microsoft/unixcoder-base/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/unixcoder-base/resolve/main/vocab.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/unixcoder-base/5604afdc964f6c53782a6813140ade5216b99006/vocab.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/resolve-cache/models/microsoft/unixcoder-base/5604afdc964f6c53782a6813140ade5216b99006/vocab.json "HTTP/1.1 200 OK"


...

merges.txt: 0.00B [00:00, ?B/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/unixcoder-base/resolve/main/tokenizer.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/unixcoder-base/resolve/main/added_tokens.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/unixcoder-base/resolve/main/special_tokens_map.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/unixcoder-base/5604afdc964f6c53782a6813140ade5216b99006/special_tokens_map.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/resolve-cache/models/microsoft/unixcoder-base/5604afdc964f6c53782a6813140ade5216b99006/special_tokens_map.json "HTTP/1.1 200 OK"


special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/unixcoder-base/resolve/main/chat_template.jinja "HTTP/1.1 404 Not Found"


Dataset Split Results
Total: 199960
Train: 163967
Val: 15997
Test: 19996
Train source counts: {'unknown': 163967}
Val source counts: {'unknown': 15997}
Test source counts: {'unknown': 19996}


In [6]:
config = RobertaConfig.from_pretrained(args.model_name_or_path)
config.num_labels = 2
encoder = RobertaModel.from_pretrained(args.model_name_or_path, config=config)
model = SimpleModel(encoder, config)
model.to(args.device)

best_epoch, best_val_acc, history = train(model, train_dataset, val_dataset, args)

model.load_state_dict(torch.load(args.best_model_path, map_location=args.device))
final_metrics = evaluate(model, test_dataset, args, tag="Final Test", quiet=False)

np.save('/kaggle/working/unixcoder_text_only_test_probs.npy', final_metrics['probs'])
np.save('/kaggle/working/unixcoder_text_only_test_labels.npy', final_metrics['labels'])

with open(args.results_path, 'w', encoding='utf-8') as f:
    f.write('Model: UniXcoder text-only\n')
    f.write('Split: stratified train/val/test\n')
    f.write(f'Seed: {args.seed}\n')
    f.write(f'Train size: {len(train_dataset)}\n')
    f.write(f'Val size: {len(val_dataset)}\n')
    f.write(f'Test size: {len(test_dataset)}\n')
    f.write(f'Epoch ceiling: {args.num_train_epochs}\n')
    f.write(f'Patience: {args.patience}\n')
    f.write(f'Best epoch: {best_epoch}\n')
    f.write(f'Best val accuracy: {best_val_acc:.4f}\n')
    f.write(f'Test Accuracy: {final_metrics["acc"]:.4f}\n')
    f.write(f'Test ROC-AUC: {final_metrics["roc_auc"]:.4f}\n')
    f.write(f'Test PR-AUC: {final_metrics["pr_auc"]:.4f}\n')
    f.write(f'FN: {final_metrics["fn"]}\n')
    f.write(f'FP: {final_metrics["fp"]}\n')
    f.write(f'Split indices file: {args.split_indices_path}\n')
    f.write(f'Split summary file: {args.split_summary_path}\n')
    f.write(f'Training history file: {args.history_path}\n')

print(f"Saved results to {args.results_path}")
print(f"Best epoch selected by early stopping: {best_epoch}")


INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/unixcoder-base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/unixcoder-base/5604afdc964f6c53782a6813140ade5216b99006/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/unixcoder-base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/unixcoder-base/5604afdc964f6c53782a6813140ade5216b99006/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/unixcoder-base/resolve/main/adapter_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/unixcoder-base/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/unixcoder-base/resolve/main/model.safetensors

pytorch_model.bin:   0%|          | 0.00/504M [00:00<?, ?B/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/unixcoder-base/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/microsoft/unixcoder-base "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/microsoft/unixcoder-base/commits/main "HTTP/1.1 200 OK"


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/microsoft/unixcoder-base/discussions?p=0 "HTTP/1.1 200 OK"
RobertaModel LOAD REPORT from: microsoft/unixcoder-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/microsoft/unixcoder-base/commits/refs%2Fpr%2F8 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/unixcoder-base/resolve/refs%2Fpr%2F8/model.safetensors.index.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/unixcoder-base/resolve/refs%2Fpr%2F8/model.safetensors "HTTP/1.1 302 Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/microsoft/unixcoder-base/xet-read-token/558aa506226b13a7e83f66cb38c53e61b9603eac "HTTP/1

...

RESULTS (Validation Epoch 4)
Accuracy : 88.9917%
ROC-AUC : 0.9615
PR-AUC : 0.9624
FN Count : 994
FP Count : 767
----------------------------------------
              precision    recall  f1-score   support

        Safe     0.8788    0.9038    0.8911      7972
        Vuln     0.9016    0.8761    0.8887      8025

    accuracy                         0.8899     15997
   macro avg     0.8902    0.8900    0.8899     15997
weighted avg     0.8902    0.8899    0.8899     15997

Confusion Matrix:
[[7205  767]
 [ 994 7031]]
New best model saved at epoch 4 with validation accuracy 0.8899


Evaluating Validation Epoch 5: 100%|██████████| 500/500 [05:48<00:00,  1.44it/s]


RESULTS (Validation Epoch 5)
Accuracy : 88.8479%
ROC-AUC : 0.9584
PR-AUC : 0.9590
FN Count : 930
FP Count : 854
----------------------------------------
              precision    recall  f1-score   support

        Safe     0.8844    0.8929    0.8886      7972
        Vuln     0.8926    0.8841    0.8883      8025

    accuracy                         0.8885     15997
   macro avg     0.8885    0.8885    0.8885     15997
weighted avg     0.8885    0.8885    0.8885     15997

Confusion Matrix:
[[7118  854]
 [ 930 7095]]
No validation improvement. Patience 1/2


Evaluating Final Test: 100%|██████████| 625/625 [07:14<00:00,  1.44it/s]

RESULTS (Final Test)
Accuracy : 89.0778%
ROC-AUC : 0.9622
PR-AUC : 0.9636
FN Count : 1238
FP Count : 946
----------------------------------------
              precision    recall  f1-score   support

        Safe     0.8792    0.9050    0.8919      9953
        Vuln     0.9030    0.8767    0.8897     10043

    accuracy                         0.8908     19996
   macro avg     0.8911    0.8908    0.8908     19996
weighted avg     0.8911    0.8908    0.8908     19996

Confusion Matrix:
[[9007  946]
 [1238 8805]]
Saved results to saved_models_unixcoder/unixcoder_text_only_results.txt
Best epoch selected by early stopping: 4
